#### Version 1

In [1]:
import json
import math
import os

AMPDAMP_REPORT_DIR = "reports/8Q-heisen-ampdamp"
DEPH_REPORT_DIR = "reports/8Q-heisen-deph"
OUTPUT_DIR = "reports"

# Set to False to show only the mean in the Error Rate column (no \pm std)
SHOW_ERROR_STD = True

# Set to True to append accuracy_vs_baseline.accuracy_pct in parens next to
# the error rate, e.g. "$0.190 (98.557\%)$". Mutually exclusive with
# SHOW_ERROR_STD — both together would clutter the error cell.
INCLUDE_ACCURACY = False

if SHOW_ERROR_STD and INCLUDE_ACCURACY:
    raise ValueError(
        "SHOW_ERROR_STD and INCLUDE_ACCURACY cannot both be True — "
        "showing std and accuracy together clutters the error rate cell. "
        "Choose one."
    )


def fmt_mean_std(mean, std):
    return f"${mean:.3f} \\pm {std:.3f}$"


def fmt_error_rate(mean, std, accuracy_pct=None, show_std=SHOW_ERROR_STD,
                    include_accuracy=INCLUDE_ACCURACY):
    if show_std and include_accuracy:
        # Defensive: should never happen given the module-level guard, but
        # this function may be called directly with explicit args too.
        raise ValueError("show_std and include_accuracy cannot both be True")
    if show_std:
        return f"${mean:.3f} \\pm {std:.3f}$"
    if include_accuracy:
        if accuracy_pct is None:
            raise ValueError("include_accuracy=True requires accuracy_pct")
        return f"${mean:.3f} ({accuracy_pct:.3f}\\%)$"
    return f"${mean:.3f}$"


def fmt_sci(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_overhead(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_int(value):
    return f"{int(round(value))}"


def load_export(path):
    with open(path) as f:
        data = json.load(f)
    rows_by_order = {int(r["order"]): r for r in data["rows"]}
    return data, rows_by_order


def build_noise_free_row(ad_data):
    mean = ad_data["noise_free_mean"]
    std = ad_data["noise_free_std"]
    return (
        r"\textbf{Noise-free VQE estimation} & -- & "
        f"{fmt_mean_std(mean, std)} & -- & -- & -- & -- & -- & -- \\\\"
    )


def build_unmitigated_block(deph_data, ad_data):
    return [
        r"\multirow{2}{64pt}{\textbf{Unmitigated VQE estimation}} & Dephasing & "
        f"{fmt_mean_std(deph_data['unmitigated_mean'], deph_data['unmitigated_std'])} "
        r"& -- & -- & -- & -- & -- & -- \\",
        r" & AD & "
        f"{fmt_mean_std(ad_data['unmitigated_mean'], ad_data['unmitigated_std'])} "
        r"& -- & -- & -- & -- & -- & -- \\",
    ]


def build_zne_block(order, d, a, show_error_std=SHOW_ERROR_STD,
                     include_accuracy=INCLUDE_ACCURACY):
    if show_error_std and include_accuracy:
        raise ValueError("show_error_std and include_accuracy cannot both be True")

    for field in ("depth", "gate", "overhead"):
        if not math.isclose(d[field], a[field], rel_tol=1e-9):
            raise ValueError(
                f"Order {order}: '{field}' differs between Dephasing "
                f"({d[field]}) and AD ({a[field]}) — expected identical "
                f"values for this classical quantity."
            )

    overhead_str = fmt_overhead(d["overhead"])
    depth_str = fmt_int(d["depth"])
    gate_str = fmt_int(d["gate"])

    d_acc = d["accuracy_vs_baseline"]["accuracy_pct"] if include_accuracy else None
    a_acc = a["accuracy_vs_baseline"]["accuracy_pct"] if include_accuracy else None

    return [
        r"\multirow{2}{*}{\textbf{ZNE of order " + str(order) + r"}} & "
        r"Dephasing & "
        f"{fmt_mean_std(d['zne_mean'], d['zne_std'])} & "
        f"{fmt_error_rate(d['error_rate']['error_rate'], d['error_rate']['error_rate_std'], d_acc, show_error_std, include_accuracy)} & "
        f"{fmt_sci(d['runtime'])} & "
        f"{fmt_sci(d['comp_cost'])} & "
        r"\multirow{2}{*}{" + overhead_str + r"} & "
        r"\multirow{2}{*}{" + depth_str + r"} & "
        r"\multirow{2}{*}{" + gate_str + r"} \\",
        r" & AD & "
        f"{fmt_mean_std(a['zne_mean'], a['zne_std'])} & "
        f"{fmt_error_rate(a['error_rate']['error_rate'], a['error_rate']['error_rate_std'], a_acc, show_error_std, include_accuracy)} & "
        f"{fmt_sci(a['runtime'])} & "
        f"{fmt_sci(a['comp_cost'])} & & & \\\\",
    ]


def build_table_body(deph_path, ad_path, show_error_std=SHOW_ERROR_STD,
                      include_accuracy=INCLUDE_ACCURACY):
    if show_error_std and include_accuracy:
        raise ValueError("show_error_std and include_accuracy cannot both be True")

    deph_data, deph_rows = load_export(deph_path)
    ad_data, ad_rows = load_export(ad_path)
    orders = sorted(set(deph_rows) & set(ad_rows))
    lines = [r"\hline"]
    lines.append(build_noise_free_row(ad_data))
    lines.append(r"\hline")
    lines.extend(build_unmitigated_block(deph_data, ad_data))
    for order in orders:
        lines.append(r"\hline")
        lines.extend(
            build_zne_block(order, deph_rows[order], ad_rows[order], show_error_std, include_accuracy)
        )
    lines.append(r"\hline")
    return "\n".join(lines)


if __name__ == "__main__":
    deph_path = os.path.join(DEPH_REPORT_DIR, "zne_multivar_dephasing.json")
    ad_path = os.path.join(AMPDAMP_REPORT_DIR, "zne_multivar_ad.json")
    body = build_table_body(deph_path, ad_path)
    out_path = os.path.join(OUTPUT_DIR, "zne_multivar_table_body.tex")
    with open(out_path, "w") as f:
        f.write(body + "\n")
    print(body)
    print("%" * 80)
    print(f"\nWritten to {out_path}")

\hline
\textbf{Noise-free VQE estimation} & -- & $-13.198 \pm 0.052$ & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{64pt}{\textbf{Unmitigated VQE estimation}} & Dephasing & $-10.067 \pm 0.116$ & -- & -- & -- & -- & -- & -- \\
 & AD & $-11.592 \pm 0.131$ & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 1}} & Dephasing & $-12.289 \pm 0.087$ & $0.909 \pm 0.101$ & $1.786 \times 10^{11}$ & $3.918 \times 10^{15}$ & \multirow{2}{*}{$1.750 \times 10^{1}$} & \multirow{2}{*}{70} & \multirow{2}{*}{174} \\
 & AD & $-13.008 \pm 0.076$ & $0.190 \pm 0.092$ & $1.591 \times 10^{11}$ & $3.918 \times 10^{15}$ & & & \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 2}} & Dephasing & $-12.856 \pm 0.062$ & $0.342 \pm 0.081$ & $1.136 \times 10^{13}$ & $4.193 \times 10^{15}$ & \multirow{2}{*}{$3.514 \times 10^{2}$} & \multirow{2}{*}{236} & \multirow{2}{*}{616} \\
 & AD & $-13.134 \pm 0.055$ & $0.064 \pm 0.075$ & $1.012 \times 10^{13}$ & $4.160 \times 10^{15}$ & & & \\
\hline
\mu

#### Version 2

In [2]:
import json
import math
import os

AMPDAMP_REPORT_DIR = "reports/8Q-heisen-ampdamp"
DEPH_REPORT_DIR = "reports/8Q-heisen-deph"
OUTPUT_DIR = "reports"

# Set to False to show only the mean in the Error Rate / Accuracy columns (no \pm std)
SHOW_ERROR_STD = False

# Set to True to add an Accuracy column right after the Error Rate column
INCLUDE_ACCURACY = True


def fmt_mean_std(mean, std):
    return f"${mean:.3f} \\pm {std:.3f}$"


def fmt_error_rate(mean, std, show_std=SHOW_ERROR_STD):
    if show_std:
        return f"${mean:.3f} \\pm {std:.3f}$"
    return f"${mean:.3f}$"


def fmt_accuracy(mean, std, show_std=SHOW_ERROR_STD):
    if show_std:
        return f"${mean:.3f} \\pm {std:.3f}$"
    return f"${mean:.3f}$"


def fmt_sci(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_overhead(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_int(value):
    return f"{int(round(value))}"


def load_export(path):
    with open(path) as f:
        data = json.load(f)
    rows_by_order = {int(r["order"]): r for r in data["rows"]}
    return data, rows_by_order


def build_noise_free_row(ad_data, include_accuracy=INCLUDE_ACCURACY):
    mean = ad_data["noise_free_mean"]
    std = ad_data["noise_free_std"]
    # Fixed trailing columns: Runtime, Comp Cost, Overhead, Depth, Gate = 5 cells,
    # regardless of whether the Accuracy column is included.
    cols = [
        r"\textbf{Noise-free VQE estimation}",
        "--",
        fmt_mean_std(mean, std),
        "--",
    ]
    if include_accuracy:
        cols.append("--")
    cols.extend(["--", "--", "--", "--", "--"])
    return " & ".join(cols) + r" \\"


def build_unmitigated_block(deph_data, ad_data, include_accuracy=INCLUDE_ACCURACY):
    deph_cols = [
        r"\multirow{2}{64pt}{\textbf{Unmitigated VQE estimation}}",
        "Dephasing",
        fmt_mean_std(deph_data["unmitigated_mean"], deph_data["unmitigated_std"]),
        "--",
    ]
    if include_accuracy:
        deph_cols.append("--")
    deph_cols.extend(["--", "--", "--", "--", "--"])

    ad_cols = [
        "",
        "AD",
        fmt_mean_std(ad_data["unmitigated_mean"], ad_data["unmitigated_std"]),
        "--",
    ]
    if include_accuracy:
        ad_cols.append("--")
    ad_cols.extend(["--", "--", "--", "--", "--"])

    return [
        " & ".join(deph_cols) + r" \\",
        " & ".join(ad_cols) + r" \\",
    ]


def build_zne_block(order, d, a, show_error_std=SHOW_ERROR_STD, include_accuracy=INCLUDE_ACCURACY):
    for field in ("depth", "gate", "overhead"):
        if not math.isclose(d[field], a[field], rel_tol=1e-9):
            raise ValueError(
                f"Order {order}: '{field}' differs between Dephasing "
                f"({d[field]}) and AD ({a[field]}) — expected identical "
                f"values for this classical quantity."
            )

    overhead_str = fmt_overhead(d["overhead"])
    depth_str = fmt_int(d["depth"])
    gate_str = fmt_int(d["gate"])

    def error_and_accuracy_cols(row_data):
        cols = [
            fmt_error_rate(
                row_data["error_rate"]["error_rate"],
                row_data["error_rate"]["error_rate_std"],
                show_error_std,
            )
        ]
        if include_accuracy:
            acc = row_data["accuracy_vs_baseline"]
            cols.append(
                fmt_accuracy(acc["accuracy_pct"], acc["accuracy_std_pct"], show_error_std)
            )
        return cols

    deph_cols = (
        [r"\multirow{2}{*}{\textbf{ZNE of order " + str(order) + r"}}", "Dephasing",
         fmt_mean_std(d["zne_mean"], d["zne_std"])]
        + error_and_accuracy_cols(d)
        + [fmt_sci(d["runtime"]), fmt_sci(d["comp_cost"]),
           r"\multirow{2}{*}{" + overhead_str + r"}",
           r"\multirow{2}{*}{" + depth_str + r"}",
           r"\multirow{2}{*}{" + gate_str + r"}"]
    )

    ad_cols = (
        ["", "AD", fmt_mean_std(a["zne_mean"], a["zne_std"])]
        + error_and_accuracy_cols(a)
        + [fmt_sci(a["runtime"]), fmt_sci(a["comp_cost"]), "", "", ""]
    )

    return [
        " & ".join(deph_cols) + r" \\",
        " & ".join(ad_cols) + r" \\",
    ]


def build_table_body(deph_path, ad_path, show_error_std=SHOW_ERROR_STD, include_accuracy=INCLUDE_ACCURACY):
    deph_data, deph_rows = load_export(deph_path)
    ad_data, ad_rows = load_export(ad_path)
    orders = sorted(set(deph_rows) & set(ad_rows))

    lines = [r"\hline"]
    lines.append(build_noise_free_row(ad_data, include_accuracy))
    lines.append(r"\hline")
    lines.extend(build_unmitigated_block(deph_data, ad_data, include_accuracy))
    for order in orders:
        lines.append(r"\hline")
        lines.extend(
            build_zne_block(order, deph_rows[order], ad_rows[order], show_error_std, include_accuracy)
        )
    lines.append(r"\hline")
    return "\n".join(lines)


if __name__ == "__main__":
    deph_path = os.path.join(DEPH_REPORT_DIR, "zne_multivar_dephasing.json")
    ad_path = os.path.join(AMPDAMP_REPORT_DIR, "zne_multivar_ad.json")
    body = build_table_body(deph_path, ad_path)
    out_path = os.path.join(OUTPUT_DIR, "zne_multivar_table_body.tex")
    with open(out_path, "w") as f:
        f.write(body + "\n")
    print(body)
    print("%" * 80)
    print(f"\nWritten to {out_path}")

\hline
\textbf{Noise-free VQE estimation} & -- & $-13.198 \pm 0.052$ & -- & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{64pt}{\textbf{Unmitigated VQE estimation}} & Dephasing & $-10.067 \pm 0.116$ & -- & -- & -- & -- & -- & -- & -- \\
 & AD & $-11.592 \pm 0.131$ & -- & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 1}} & Dephasing & $-12.289 \pm 0.087$ & $0.909$ & $93.116$ & $1.786 \times 10^{11}$ & $3.918 \times 10^{15}$ & \multirow{2}{*}{$1.750 \times 10^{1}$} & \multirow{2}{*}{70} & \multirow{2}{*}{174} \\
 & AD & $-13.008 \pm 0.076$ & $0.190$ & $98.557$ & $1.591 \times 10^{11}$ & $3.918 \times 10^{15}$ &  &  &  \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 2}} & Dephasing & $-12.856 \pm 0.062$ & $0.342$ & $97.407$ & $1.136 \times 10^{13}$ & $4.193 \times 10^{15}$ & \multirow{2}{*}{$3.514 \times 10^{2}$} & \multirow{2}{*}{236} & \multirow{2}{*}{616} \\
 & AD & $-13.134 \pm 0.055$ & $0.064$ & $99.515$ & $1.012 \times 10^{13}$ & $4.160 \times 10^{15